## Libraries

In [1]:
import requests
import json
import os
import sys
import pandas as pd
import geopandas as gpd
import time
from IPython.display import Markdown
from shapely import wkb

## Functions

In [2]:
sys.path.append(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness")

In [3]:
# API count building number by geometry
def post_count(geom, filter="type:way and building=*", time="2008-01-01/2025-01-01/P1M", path=None, filename=None):
    '''
    A function that extracts cummulative number of buildings (building tags) in a specific geometry from OSM using the Ohsome API, and returns all data in a json format.

    The function recieves 3 strings:
        1) **geom** - simplified geometry parameter
        2) **filter** - filter parameter, defaultly extracting all building tags from way objects
        3) **time** - ISO-8621 timestamp. Defaultly extracting all data between 01/2008-01/2025 in monthly intervals
    
    If the user wants to save the data, there are two additional parameters:
        1) **path** - string to path on computer
        2) **filename** - string with requested filename

    Dependencies:
    * requests
    * json
    * os
    * pandas as pd
    '''
    # Ensuring needed libraries are imported
    import requests
    import json
    import os
    import pandas as pd

    url = "https://api.ohsome.org/v1/elements/count/groupBy/boundary" # URL for API

    # Defining parameters for extract
    params = {
        "bpolys": json.dumps({
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "properties": {"id": "region"},
                "geometry": geom
            }]
        }),
        "filter": filter,
        "time": time,
        "format": "json"
    }


    response = requests.post(url, data=params) # Creating request

    if response.status_code == 200:
        print("Succesfully extracted counts")
        data = response.json() # data extract

        # Saving extract to file if requested:
        if path and filename:
            os.makedirs(path, exist_ok=True)  # create directory if it doesn't exist
            file_path = os.path.join(path, filename)
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"Data saved to {file_path}")

        return pd.json_normalize(data['groupByResult'][0]['result'])
    
    # Print errors if recieved for debugging 
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

In [4]:
# API post bulding area function by geometry
def post_area(geom, filter="type:way and building=*", time="2008-01-01/2025-01-01/P1M", path=None, filename=None):
    '''
    A function that extracts aggregated area of buildings (highway tags) in a specific geometry from OSM using the Ohsome API, and returns all data in a json format.

    The function recieves 3 strings:
        1) **geom** - simplified geometry parameter
        2) **filter** - filter parameter, defaultly extracting all highway tags from way objects
        3) **time** - ISO-8621 timestamp. Defaultly extracting all data between 01/2008-01/2025 in monthly intervals
    
    If the user wants to save the data, there are two additional parameters:
        1) **path** - string to path on computer
        2) **filename** - string with requested filename
    
    Dependencies:
    * requests
    * json
    * os
    * pandas as pd
    '''
    # Ensuring needed libraries are imported
    import requests
    import json
    import os
    import pandas as pd

    url = "https://api.ohsome.org/v1/elements/area/groupBy/boundary" # URL for API

    # Defining parameters for extract
    params = {
        "bpolys": json.dumps({
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "properties": {"id": "region"},
                "geometry": geom
            }]
        }),
        "filter": filter,
        "time": time,
        "format": "json"
    }
    
    response = requests.post(url, data=params) # Creating request

    if response.status_code == 200:
        print("Succesfully extracted areas")
        data = response.json() # data extract

        # Saving extract to file if requested:
        if path and filename:
            os.makedirs(path, exist_ok=True)  # create directory if it doesn't exist
            file_path = os.path.join(path, filename)
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"Data saved to {file_path}")

        return pd.json_normalize(data['groupByResult'][0]['result'])
    
    # Print errors if recieved for debugging 
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

## Extract areas from Parquet

In [5]:
regions = pd.read_parquet(r"H:\.shortcut-targets-by-id\1vC82Zl3hhtFy63TpICgdDiHqTHm5dv0h\OSM Projects\Data\overture_map_results\region_area-2025-10-22.0_dataset_counts_strtree.parquet")

In [6]:
compact_regions = regions[['geometry', 'bbox', 'country', 'region', 'openstreetmap', 'esri_community_maps', 'instituto_geografico_nacional_espana',
                     'google_open_buildings', 'microsoft_ml_buildings', 'doi_10_5281_zenodo_8174931', 'usgs_lidar']].copy()

compact_regions.head()

,geometry,bbox,country,region,openstreetmap,esri_community_maps,instituto_geografico_nacional_espana,google_open_buildings,microsoft_ml_buildings,doi_10_5281_zenodo_8174931,usgs_lidar
0,b'\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...,"{'xmax': 12.516959190368652, 'xmin': 12.469887...",SM,SM-04,387,0,0,0,96,0,0
1,b'\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...,"{'xmax': 29.028268814086914, 'xmin': 28.595144...",MD,MD-RE,6246,0,0,0,26544,0,0
2,b'\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...,"{'xmax': 14.878803253173828, 'xmin': 14.695601...",SI,SI-030,2365,0,0,0,201,0,0
3,b'\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...,"{'xmax': -69.94135284423828, 'xmin': -77.82595...",PE,PE-LOR,177349,0,0,136045,105941,0,0
4,b'\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...,"{'xmax': -5.333332538604736, 'xmin': -9.149688...",MR,MR-01,58126,0,0,186933,107527,0,0


In [7]:
compact_regions['geometry'] = compact_regions['geometry'].apply(wkb.loads) # Fixing geometry column

In [8]:
compact_regions.head()

,geometry,bbox,country,region,openstreetmap,esri_community_maps,instituto_geografico_nacional_espana,google_open_buildings,microsoft_ml_buildings,doi_10_5281_zenodo_8174931,usgs_lidar
0,"POLYGON ((12.5155494 43.9398893, 12.516219 43....","{'xmax': 12.516959190368652, 'xmin': 12.469887...",SM,SM-04,387,0,0,0,96,0,0
1,"POLYGON ((28.8541043 47.8171267, 28.8537056 47...","{'xmax': 29.028268814086914, 'xmin': 28.595144...",MD,MD-RE,6246,0,0,0,26544,0,0
2,"POLYGON ((14.7043091 46.2869076, 14.7047036 46...","{'xmax': 14.878803253173828, 'xmin': 14.695601...",SI,SI-030,2365,0,0,0,201,0,0
3,"POLYGON ((-70.17954450000001 -4.0214807, -70.1...","{'xmax': -69.94135284423828, 'xmin': -77.82595...",PE,PE-LOR,177349,0,0,136045,105941,0,0
4,"POLYGON ((-6.3790366 23.1076079, -7.9877222 19...","{'xmax': -5.333332538604736, 'xmin': -9.149688...",MR,MR-01,58126,0,0,186933,107527,0,0


## Test simplified geometry extract

In [9]:
simplified_geom = compact_regions.geometry.iloc[0].simplify(0.01).__geo_interface__

In [10]:
test = post_count(simplified_geom,
           time="2008-01-22/2025-10-22/P1M")

Succesfully extracted counts


In [11]:
display(test)

,timestamp,value
0,2008-01-22T00:00:00Z,0.0
1,2008-02-22T00:00:00Z,0.0
2,2008-03-22T00:00:00Z,0.0
3,2008-04-22T00:00:00Z,0.0
4,2008-05-22T00:00:00Z,0.0
...,...,...
209,2025-06-22T00:00:00Z,379.0
210,2025-07-22T00:00:00Z,379.0
211,2025-08-22T00:00:00Z,379.0
212,2025-09-22T00:00:00Z,379.0


In [12]:
test2 = post_area(simplified_geom,
           time="2008-01-22/2025-10-22/P1M")

Succesfully extracted areas


In [13]:
display(test2)

,timestamp,value
0,2008-01-22T00:00:00Z,0.00
1,2008-02-22T00:00:00Z,0.00
2,2008-03-22T00:00:00Z,0.00
3,2008-04-22T00:00:00Z,0.00
4,2008-05-22T00:00:00Z,0.00
...,...,...
209,2025-06-22T00:00:00Z,146674.28
210,2025-07-22T00:00:00Z,146674.28
211,2025-08-22T00:00:00Z,146674.28
212,2025-09-22T00:00:00Z,146674.28


In [9]:
# Fix DF to become GDF
compact_regions = gpd.GeoDataFrame(
    compact_regions,
    geometry=compact_regions['geometry']
)

In [10]:
compact_regions['geometry'] = compact_regions.geometry.simplify(0.01) # Simplify all geometries for faster processing

## Extract data

In [11]:
germany = compact_regions[compact_regions['country'] == 'DE']
portugal = compact_regions[compact_regions['country'] == 'PT']
france = compact_regions[compact_regions['country'] == 'FR']

In [17]:
portugal_counts = []

for _, row in portugal.iterrows():

    geom = row.geometry.__geo_interface__

    df = post_count(
        geom=geom,
        time="2008-01-22/2025-10-22/P1M"
    )

    if df is None:
        continue

    df['ISO'] = row['country']
    df['region'] = row['region']

    portugal_counts.append(df)

portugal_counts_neat = pd.concat(portugal_counts, ignore_index=True)

Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts
Succesfully extracted counts


In [ ]:
# save data
portugal_counts_neat.to_csv(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\geometry_clipped_data\portugal_bld_counts_by_region.csv", index=False)

In [21]:
portugal_areas = []

for _, row in portugal.iterrows():

    geom = row.geometry.__geo_interface__

    df = post_area(
        geom=geom,
        time="2008-01-22/2025-10-22/P1M"
    )

    if df is None:
        continue

    df['ISO'] = row['country']
    df['region'] = row['region']

    portugal_areas.append(df)

portugal_areas_neat = pd.concat(portugal_areas, ignore_index=True)

Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas
Succesfully extracted areas


In [22]:
# save data
portugal_areas_neat.to_csv(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\geometry_clipped_data\portugal_bld_areas_by_region.csv", index=False)

In [ ]:
germany_counts = []
germany_areas = []

for row in germany.itertuples(index=False):

    geom = row.geometry.__geo_interface__

    df1 = post_count(
        geom=geom,
        time="2008-01-22/2025-10-22/P1M"
    )

    df2 = post_area(
        geom=geom,
        time="2008-01-22/2025-10-22/P1M"
    )

    if df1 is None or df2 is None:
        continue

    for df in (df1, df2):
        df['ISO'] = row.country
        df['region'] = row.region

    germany_counts.append(df1)
    germany_areas.append(df2)

germany_counts_neat = pd.concat(germany_counts, ignore_index=True)
germany_areas_neat = pd.concat(germany_areas, ignore_index=True)

In [ ]:
# save data
germany_counts_neat.to_csv(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\geometry_clipped_data\germany_bld_counts_by_region.csv", index=False)
germany_areas_neat.to_csv(r"C:\Users\eliya\Documents\עבודה\עוזר מחקר\4. פרויקטים\1. feature completness\OSM-Feature-Completeness\1) Data\4) overture comparisson\geometry_clipped_data\germany_bld_areas_by_region.csv", index=False)

In [ ]:
# france_counts = []
# france_areas = []

# for row in france.itertuples(index=False):

#     geom = row.geometry.__geo_interface__

#     df1 = post_count(
#         geom=geom,
#         time="2008-01-22/2025-10-22/P1M"
#     )

#     df2 = post_area(
#         geom=geom,
#         time="2008-01-22/2025-10-22/P1M"
#     )

#     if df1 is None or df2 is None:
#         continue

#     for df in (df1, df2):
#         df['ISO'] = row.country
#         df['region'] = row.region

#     france_counts.append(df1)
#     france_areas.append(df2)

# france_counts_neat = pd.concat(france_counts, ignore_index=True)
# france_areas_neat = pd.concat(france_areas, ignore_index=True)

Error 500: {
  "timestamp" : "2026-04-16T16:21:00.394+00:00",
  "status" : 500,
  "error" : "Internal Server Error",
  "path" : "/elements/count/groupBy/boundary"
}
Error 500: {
  "timestamp" : "2026-04-16T16:21:01.602+00:00",
  "status" : 500,
  "error" : "Internal Server Error",
  "path" : "/elements/area/groupBy/boundary"
}
